In [0]:
#split by prediction month, with a three-month reporting buffer as a project assumption:

#Training	    January 2017–September 2020
#Validation	    January–September 2022
#Final test	    January 2024–March 2025

In [0]:
from pyspark.sql import functions as F

In [0]:
#Load Gold and add a few useful features
from pyspark.sql import functions as F

gold = spark.table("workspace.mortgage_risk.gold_mortgage_12m")

# Original loan age, which does not reset after modification
gold = gold.withColumn("original_age_months",
    F.months_between("prediction_month", "first_payment_date").cast("int") + 1)

# Use one versus multiple borrowers consistently across vintages
gold = gold.withColumn("multiple_borrowers",F.when(F.col("borrowers") > 1, 1).when(F.col("borrowers") == 1, 0))

gold = gold.withColumn(
    "rate_change",
    F.col("current_rate") - F.col("original_rate")
)

In [0]:
# Select the predictors explicitly
numeric_features = [
    "fico",
    "ltv",
    "additional_lien_ltv",
    "dti",
    "original_upb",
    "original_rate",
    "original_term",
    "mi_pct",
    "multiple_borrowers",
    "units",
    "original_age_months",
    "loan_age",
    "remaining_term",
    "current_rate",
    "rate_change",
    "estimated_ltv",
    "balance_ratio",
    "delinquency_months",
    "late_months_3m",
    "late_months_6m",
    "max_delinquency_3m",
    "max_delinquency_6m"
]

categorical_features = [
    "purpose",
    "occupancy",
    "property_type",
    "state",
    "channel"
]

data = gold.filter(F.col("label_12m").isNotNull())

data = data.select(
    "loan_id",
    "prediction_month",
    "label_12m",
    "label_available_month",
    *numeric_features,
    *categorical_features
)

for column in numeric_features:
    data = data.withColumn(column, F.col(column).cast("double"))

In [0]:
# Create the time splits
reporting_lag_months = 3

# Require the full horizon to mature
data = data.withColumn(
    "horizon_available_month",
    F.add_months(
        F.col("prediction_month"),
        12 + reporting_lag_months
    )
)

train_fit_month = F.to_date(F.lit("2022-01-01"))
final_fit_month = F.to_date(F.lit("2024-01-01"))

train = data.filter(
    F.col("prediction_month").between("2017-01-01", "2020-12-01")
    & (F.col("horizon_available_month") < train_fit_month)
)

validation = data.filter(
    F.col("prediction_month").between("2022-01-01", "2022-12-01")
    & (F.col("horizon_available_month") < final_fit_month)
)

test = data.filter(
    F.col("prediction_month").between("2024-01-01", "2025-03-01")
)

In [0]:
#Check sizes, event counts, and dates
train_summary = train.agg(
    F.lit("train").alias("dataset"),
    F.count("*").alias("rows"),
    F.countDistinct("loan_id").alias("unique_loans"),
    F.sum("label_12m").alias("positive_rows"),
    F.avg("label_12m").alias("event_rate"),
    F.min("prediction_month").alias("first_month"),
    F.max("prediction_month").alias("last_month"),
    F.max("horizon_available_month").alias("latest_horizon_available")
)

validation_summary = validation.agg(
    F.lit("validation").alias("dataset"),
    F.count("*").alias("rows"),
    F.countDistinct("loan_id").alias("unique_loans"),
    F.sum("label_12m").alias("positive_rows"),
    F.avg("label_12m").alias("event_rate"),
    F.min("prediction_month").alias("first_month"),
    F.max("prediction_month").alias("last_month"),
    F.max("horizon_available_month").alias("latest_horizon_available")
)

test_summary = test.agg(
    F.lit("test").alias("dataset"),
    F.count("*").alias("rows"),
    F.countDistinct("loan_id").alias("unique_loans"),
    F.sum("label_12m").alias("positive_rows"),
    F.avg("label_12m").alias("event_rate"),
    F.min("prediction_month").alias("first_month"),
    F.max("prediction_month").alias("last_month"),
    F.max("horizon_available_month").alias("latest_horizon_available")
)

split_summary = (
    train_summary
    .unionByName(validation_summary)
    .unionByName(test_summary)
)

display(split_summary)

dataset,rows,unique_loans,positive_rows,event_rate,first_month,last_month,latest_horizon_available
train,4299547,199451,88771,0.020646593699289714,2017-01-01,2020-09-01,2021-12-01
validation,1247263,161054,6134,0.004917968383572671,2022-01-01,2022-09-01,2023-12-01
test,3306965,257418,25184,0.007615441953573745,2024-01-01,2025-03-01,2026-06-01


In [0]:
# Save the splits 
split_tables = [
    ("model_train", train),
    ("model_validation", validation),
    ("model_test", test)
]

for name, dataframe in split_tables:
    table_name = f"workspace.mortgage_risk.{name}"

    if spark.catalog.tableExists(table_name):
        print(f"{name} already exists. Skipping save.")
    else:
        dataframe.write.format("delta").mode("error").saveAsTable(table_name)
        print(f"{name} saved.")

model_train saved.
model_validation saved.
model_test saved.
